In [10]:
"""
Phase 3: Data Cleaning and Preprocessing
========================================
Inputs : data/raw/{brand}_posts_raw.csv, data/raw/{brand}_profiles_raw.csv
Outputs: data/processed/{brand}_posts_clean.csv, data/processed/{brand}_profiles_clean.csv
         data/processed/cleaning_summary.json

The cleaning pipeline mirrors Week 7's approach (Week_7_Text_Analytics_case_study)
with three Bluesky-specific additions:
    1. Extract URLs / domains / mentions / hashtags BEFORE stripping them
       (so we can use them as separate features in network and engagement analysis).
    2. Maintain three text columns — raw, cleaned, and cleaned-without-brand-tokens —
       because different downstream tasks need different preprocessing aggressiveness.
    3. Filter spam patterns specific to Bluesky (link-only posts, excessive
       repeats from the same author).
"""

import json
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
from textblob import Word

# Download required NLTK data (idempotent)
for pkg in ["stopwords", "wordnet", "punkt", "punkt_tab"]:
    try:
        nltk.data.find(f"corpora/{pkg}" if pkg != "punkt" and pkg != "punkt_tab" else f"tokenizers/{pkg}")
    except LookupError:
        nltk.download(pkg, quiet=True)

# Project paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROC_DIR = PROJECT_ROOT / "data" / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

print(f"Reading raw data from : {RAW_DIR}")
print(f"Writing processed data to: {PROC_DIR}")

Reading raw data from : /Users/shreyu/VSCODE/junk/UoN-Business-Analytics/Analytics-Specializations-and-Applications/BUSI4370_CW2/data/raw
Writing processed data to: /Users/shreyu/VSCODE/junk/UoN-Business-Analytics/Analytics-Specializations-and-Applications/BUSI4370_CW2/data/processed


In [11]:
# Load each brand's posts and profiles separately so we can track cleaning
# decisions per brand, then merge for any cross-brand analysis later.
hf_posts_raw  = pd.read_csv(RAW_DIR / "huggingface_posts_raw.csv")
rep_posts_raw = pd.read_csv(RAW_DIR / "replicate_posts_raw.csv")

hf_profiles_raw  = pd.read_csv(RAW_DIR / "huggingface_profiles_raw.csv")
rep_profiles_raw = pd.read_csv(RAW_DIR / "replicate_profiles_raw.csv")

print(f"HF posts (raw)        : {len(hf_posts_raw):>5}")
print(f"Replicate posts (raw) : {len(rep_posts_raw):>5}")
print(f"HF profiles (raw)     : {len(hf_profiles_raw):>5}")
print(f"Replicate profiles    : {len(rep_profiles_raw):>5}")

HF posts (raw)        :  1856
Replicate posts (raw) :  1109
HF profiles (raw)     :   832
Replicate profiles    :   793


In [12]:
# We extract these BEFORE cleaning so the cleaned text columns don't lose them,
# but the cleaned text doesn't carry them as noise either.
URL_RE     = re.compile(r"https?://\S+|www\.\S+|\S+\.(?:com|co|io|net|org|ai|app|dev|gg)/\S*", re.IGNORECASE)
MENTION_RE = re.compile(r"@[\w.-]+")
HASHTAG_RE = re.compile(r"#\w+")
DOMAIN_RE  = re.compile(r"(?:https?://)?(?:www\.)?([\w-]+(?:\.[\w-]+)+)", re.IGNORECASE)

def extract_features(text: str) -> dict:
    """Pull URLs, mentions, hashtags, and domains out of a post's raw text."""
    if not isinstance(text, str):
        return {"urls": [], "mentions": [], "hashtags": [], "domains": []}
    urls     = URL_RE.findall(text)
    mentions = MENTION_RE.findall(text)
    hashtags = [h.lower() for h in HASHTAG_RE.findall(text)]
    # Pull bare domains out of any URLs found
    domains  = [m.group(1).lower() for m in DOMAIN_RE.finditer(text) if m.group(1)]
    # Filter out false positives (e.g., 'e.g')
    domains  = [d for d in domains if d.count(".") >= 1 and len(d) >= 4]
    return {"urls": urls, "mentions": mentions, "hashtags": hashtags, "domains": domains}

def add_extracted_features(df: pd.DataFrame) -> pd.DataFrame:
    feats = df["text"].apply(extract_features).apply(pd.Series)
    # Store as semicolon-joined strings to round-trip safely through CSV
    df = df.copy()
    df["urls"]      = feats["urls"].apply(lambda xs: ";".join(xs))
    df["mentions"]  = feats["mentions"].apply(lambda xs: ";".join(xs))
    df["hashtags"]  = feats["hashtags"].apply(lambda xs: ";".join(xs))
    df["domains"]   = feats["domains"].apply(lambda xs: ";".join(xs))
    df["n_urls"]    = feats["urls"].apply(len)
    df["n_mentions"]= feats["mentions"].apply(len)
    df["n_hashtags"]= feats["hashtags"].apply(len)
    return df

hf_posts  = add_extracted_features(hf_posts_raw)
rep_posts = add_extracted_features(rep_posts_raw)
print("Extracted URL/mention/hashtag/domain features.")

Extracted URL/mention/hashtag/domain features.


In [13]:
# Stopword set: NLTK English + custom additions for Bluesky/AI discourse.
# 'huggingface', 'replicate' etc. are added to a *separate* set so we can
# remove them only for topic modelling (where they would dominate as the topic),
# not for general keyword frequency or sentiment.
BASE_STOPWORDS = set(stopwords.words("english"))

CUSTOM_STOPWORDS = {
    # generic web/Bluesky noise
    "rt", "amp", "via", "http", "https", "com", "www", "html",
    "bsky", "app", "post", "thread", "skeet",
    # filler
    "like", "get", "got", "go", "going", "use", "using", "used",
    "one", "two", "would", "could", "also", "really", "much", "many",
    "see", "see", "say", "said", "make", "made", "take", "took",
    "want", "wanted", "even", "still", "well", "way", "things", "thing",
    "today", "yesterday", "tomorrow", "now", "ll", "ve", "re",
}

# Brand tokens — only stripped for topic-modelling text variant
BRAND_STOPWORDS = {
    "huggingface", "hugging", "face", "hf", "hfco",
    "replicate", "replicates", "replicating", "replication",
    "replicatecom", "rim", "r8im",
}

ALL_STOPWORDS         = BASE_STOPWORDS | CUSTOM_STOPWORDS
ALL_STOPWORDS_NO_BRAND = ALL_STOPWORDS | BRAND_STOPWORDS


def clean_text(text: str) -> str:
    """
    Course-aligned aggressive clean (mirrors Week 7's cleanText):
      - strip URLs / mentions / hashtags (we already extracted them)
      - lowercase
      - remove punctuation, digits, single letters
      - normalise accented characters
    """
    if not isinstance(text, str):
        return ""
    # Bluesky-specific strips first
    text = URL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = HASHTAG_RE.sub(" ", text)
    # Course-style cleaning
    text = re.sub(r"[^\w\s]", " ", text)         # remove punctuation
    text = re.sub(r"\d+", " ", text)              # remove digits
    text = re.sub(r"\b[a-zA-Z]\b", " ", text)     # remove single letters
    text = text.lower()
    text = re.sub(u"[àáâãäå]", "a", text)
    text = re.sub(u"[èéêë]", "e", text)
    text = re.sub(u"[ìíîï]", "i", text)
    text = re.sub(u"[òóôõö]", "o", text)
    text = re.sub(u"[ùúûü]", "u", text)
    text = re.sub(u"[ț]", "t", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def remove_stopwords_lemmatize(text: str, stopword_set: set) -> str:
    """Token-level pass: drop stopwords and lemmatize what survives."""
    if not text:
        return ""
    tokens = [
        Word(tok).lemmatize()
        for tok in text.split()
        if tok not in stopword_set and len(tok) > 2
    ]
    return " ".join(tokens)


def light_clean_for_sentiment(text: str) -> str:
    """
    Lighter clean for VADER / TextBlob sentiment — keeps casing, punctuation,
    emoji-style cues, but strips URLs/mentions/hashtags as those don't carry
    affect and inflate the input length.
    """
    if not isinstance(text, str):
        return ""
    text = URL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = HASHTAG_RE.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [14]:
def clean_posts(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df.rename(columns={"text": "text_raw"})

    # Light clean — for sentiment scoring (VADER prefers natural text)
    df["text_for_sentiment"] = df["text_raw"].apply(light_clean_for_sentiment)

    # Aggressive clean — for keyword/frequency analysis
    df["text_clean_stage1"] = df["text_raw"].apply(clean_text)
    df["text_clean"] = df["text_clean_stage1"].apply(
        lambda t: remove_stopwords_lemmatize(t, ALL_STOPWORDS)
    )

    # Topic-modelling variant — same as text_clean but brand tokens removed
    df["text_clean_no_brand"] = df["text_clean_stage1"].apply(
        lambda t: remove_stopwords_lemmatize(t, ALL_STOPWORDS_NO_BRAND)
    )

    # Word counts (used in analysis: post length distribution, engagement vs length)
    df["word_count_raw"]   = df["text_raw"].fillna("").str.split().str.len()
    df["word_count_clean"] = df["text_clean"].str.split().str.len()

    # Drop the intermediate stage column to keep CSV tidy
    df = df.drop(columns=["text_clean_stage1"])
    return df

hf_posts  = clean_posts(hf_posts)
rep_posts = clean_posts(rep_posts)
print("Applied three-tier text cleaning to both brands.")
hf_posts[["text_raw", "text_clean", "text_clean_no_brand"]].head(3)

Applied three-tier text cleaning to both brands.


,text_raw,text_clean,text_clean_no_brand
0,The model ecosystem reflects this. A few domin...,model ecosystem reflects dominant llm long hal...,model ecosystem reflects dominant llm long hal...
1,🤖 mistralai/Mistral-Medium-3.5-128B · Hugging ...,mistralai mistral medium hugging face mistral ...,mistralai mistral medium mistral medium mistra...
2,The model is HHEM-2.1-Open. ~600 MB at FP32. A...,model hhem open apache hugging face input retr...,model hhem open apache input retrieved context...


In [15]:
def filter_posts(df: pd.DataFrame, brand_label: str) -> tuple[pd.DataFrame, dict]:
    """
    Apply quality filters and record what was removed at each step.
    Returns (filtered_df, audit_dict) for transparent reporting.
    """
    audit = {"start": len(df)}

    # 1. Language — keep English-only (some posts may have multiple langs flagged)
    if "langs" in df.columns:
        df = df[df["langs"].fillna("").str.contains("en", case=False, regex=False)]
    audit["after_language"] = len(df)

    # 2. Deduplicate by URI (already done in collection but defensive here)
    df = df.drop_duplicates(subset="uri", keep="first")
    audit["after_uri_dedup"] = len(df)

    # 3. Deduplicate near-identical text from the same author (likely repost-spam)
    df = df.drop_duplicates(subset=["author_did", "text_raw"], keep="first")
    audit["after_author_text_dedup"] = len(df)

    # 4. Length filter — remove posts that are essentially link-only or empty after cleaning
    df = df[df["word_count_clean"] >= 3]
    audit["after_min_length"] = len(df)

    # 5. Spam signal: same author posting >5 times with identical cleaned text
    spam_mask = df.groupby(["author_did", "text_clean"])["uri"].transform("count") > 5
    df = df[~spam_mask]
    audit["after_spam_filter"] = len(df)

    # 6. Parse timestamps now (will be needed in analysis)
    df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce", utc=True)
    df = df.dropna(subset=["created_at"])
    audit["after_timestamp_parse"] = len(df)

    df = df.reset_index(drop=True)
    audit["final"] = len(df)
    print(f"\n[{brand_label}] filtering audit:")
    for k, v in audit.items():
        print(f"  {k:<28} {v:>5}")
    return df, audit


hf_posts, hf_audit   = filter_posts(hf_posts, "Hugging Face")
rep_posts, rep_audit = filter_posts(rep_posts, "Replicate")


[Hugging Face] filtering audit:
  start                         1856
  after_language                1856
  after_uri_dedup               1856
  after_author_text_dedup       1716
  after_min_length              1621
  after_spam_filter             1621
  after_timestamp_parse         1608
  final                         1608

[Replicate] filtering audit:
  start                         1109
  after_language                1109
  after_uri_dedup               1109
  after_author_text_dedup       1107
  after_min_length              1041
  after_spam_filter             1041
  after_timestamp_parse         1032
  final                         1032


In [16]:
# === Replicate-specific brand-relevance filter ===
#
# After Phase 4 inspection, the original ambiguity filter (which kept any post
# matching either replicate.com/@replicate OR an AI keyword) admitted a meaningful
# number of false positives — posts using "replicate" as a verb in AI-adjacent
# but off-brand contexts (e.g. "no AI can replicate this art style", "model that
# may replicate at the strait of Hormuz"). We tighten the filter to require
# either an explicit brand domain reference OR co-occurrence of the brand name
# with strong product-context anchors (cog, r8.im, model deployment terms).

# Strong brand signals — direct domain/handle references
BRAND_SIGNALS = re.compile(
    r"replicate\.com|@replicate|r8\.im|replicate\.delivery|"
    r"replicate api|replicate cog|replicate gpu|replicate inference",
    flags=re.IGNORECASE,
)

# Anchor terms — co-occurrence with "replicate" strongly implies the brand
PRODUCT_ANCHORS = re.compile(
    r"\b(model|api|inference|deploy(?:ed|ing|ment)?|gpu|cog|host(?:ed|ing)?|"
    r"flux|sdxl|llama|whisper|stable\s*diffusion|"
    r"endpoint|pipeline|webhook|prediction|"
    r"fal\.ai|modal\.com|runpod|together\.ai)\b",
    flags=re.IGNORECASE,
)

# The word "replicate" used near a product anchor (within ~12 words) is brand-relevant.
# We use a simple proximity heuristic: both must appear in the same post.
REPLICATE_TOKEN = re.compile(r"\breplicate\b", flags=re.IGNORECASE)

def is_brand_relevant_replicate(text: str) -> bool:
    if not isinstance(text, str) or not text.strip():
        return False
    # Path 1: explicit brand signal — always relevant
    if BRAND_SIGNALS.search(text):
        return True
    # Path 2: contains "replicate" AND a product anchor in the same post
    if REPLICATE_TOKEN.search(text) and PRODUCT_ANCHORS.search(text):
        return True
    # Otherwise reject — the post mentions AI things but probably uses
    # "replicate" as a verb in an off-brand context.
    return False

before = len(rep_posts)
rep_posts = rep_posts[rep_posts["text_raw"].apply(is_brand_relevant_replicate)].reset_index(drop=True)
print(f"Replicate brand-relevance filter: {before} -> {len(rep_posts)} posts retained")
print(f"Removed {before - len(rep_posts)} posts that mentioned 'replicate' as a verb in off-brand contexts.")

# Update the audit
rep_audit["after_brand_relevance"] = len(rep_posts)
rep_audit["final"] = len(rep_posts)

Replicate brand-relevance filter: 1032 -> 672 posts retained
Removed 360 posts that mentioned 'replicate' as a verb in off-brand contexts.


In [17]:
def clean_profiles(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df.drop_duplicates(subset="did", keep="first")
    # Coerce numeric columns and parse account creation date
    for col in ["followers_count", "follows_count", "posts_count"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)
    df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce", utc=True)
    # Engagement-relevant ratios — pre-compute for influencer scoring later
    df["follower_following_ratio"] = np.where(
        df["follows_count"] > 0,
        df["followers_count"] / df["follows_count"],
        df["followers_count"]   # if following 0, ratio is just follower count
    )
    return df.reset_index(drop=True)

hf_profiles  = clean_profiles(hf_profiles_raw)
rep_profiles = clean_profiles(rep_profiles_raw)

# Filter profiles to those whose author actually appears in the cleaned post set
hf_profiles  = hf_profiles[hf_profiles["did"].isin(hf_posts["author_did"])].reset_index(drop=True)
rep_profiles = rep_profiles[rep_profiles["did"].isin(rep_posts["author_did"])].reset_index(drop=True)

print(f"HF profiles after alignment with cleaned posts        : {len(hf_profiles)}")
print(f"Replicate profiles after alignment with cleaned posts : {len(rep_profiles)}")

HF profiles after alignment with cleaned posts        : 770
Replicate profiles after alignment with cleaned posts : 437


In [18]:
hf_posts.to_csv(PROC_DIR / "huggingface_posts_clean.csv", index=False)
rep_posts.to_csv(PROC_DIR / "replicate_posts_clean.csv", index=False)
hf_profiles.to_csv(PROC_DIR / "huggingface_profiles_clean.csv", index=False)
rep_profiles.to_csv(PROC_DIR / "replicate_profiles_clean.csv", index=False)

# Write a transparent audit JSON — quoting this in the methodology section
# directly addresses the rubric's "summary of process" criterion.
cleaning_summary = {
    "huggingface": {
        "filtering_audit": hf_audit,
        "n_posts_final": len(hf_posts),
        "n_unique_authors_final": int(hf_posts["author_did"].nunique()),
        "n_profiles_final": len(hf_profiles),
        "date_range": [str(hf_posts["created_at"].min()), str(hf_posts["created_at"].max())],
        "median_word_count": float(hf_posts["word_count_clean"].median()),
        "reply_share": float(hf_posts["is_reply"].mean()),
    },
    "replicate": {
        "filtering_audit": rep_audit,
        "n_posts_final": len(rep_posts),
        "n_unique_authors_final": int(rep_posts["author_did"].nunique()),
        "n_profiles_final": len(rep_profiles),
        "date_range": [str(rep_posts["created_at"].min()), str(rep_posts["created_at"].max())],
        "median_word_count": float(rep_posts["word_count_clean"].median()),
        "reply_share": float(rep_posts["is_reply"].mean()),
    },
    "cleaning_steps_applied": [
        "URL/mention/hashtag extraction (preserved as separate columns)",
        "Light-clean for sentiment (URLs/mentions/hashtags stripped, casing kept)",
        "Aggressive clean for keyword/topic analysis (lowercase, no punct/digits/single letters, NLTK stopwords + custom stopwords, lemmatised via TextBlob)",
        "Brand-name stopword variant for topic modelling (text_clean_no_brand)",
        "English-language filter (langs contains 'en')",
        "URI-level deduplication",
        "Same-author/same-text deduplication (repost spam)",
        "Minimum length filter (>=3 cleaned tokens)",
        "Author-level repeated-content spam filter (>5 identical posts)",
    ],
}
with open(PROC_DIR / "cleaning_summary.json", "w") as f:
    json.dump(cleaning_summary, f, indent=2)

print("Saved processed data + cleaning_summary.json")
print(json.dumps(cleaning_summary, indent=2))

Saved processed data + cleaning_summary.json
{
  "huggingface": {
    "filtering_audit": {
      "start": 1856,
      "after_language": 1856,
      "after_uri_dedup": 1856,
      "after_author_text_dedup": 1716,
      "after_min_length": 1621,
      "after_spam_filter": 1621,
      "after_timestamp_parse": 1608,
      "final": 1608
    },
    "n_posts_final": 1608,
    "n_unique_authors_final": 770,
    "n_profiles_final": 770,
    "date_range": [
      "2025-03-07 03:32:07.171000+00:00",
      "2026-04-29 18:26:06.838000+00:00"
    ],
    "median_word_count": 16.0,
    "reply_share": 0.2922885572139303
  },
  "replicate": {
    "filtering_audit": {
      "start": 1109,
      "after_language": 1109,
      "after_uri_dedup": 1109,
      "after_author_text_dedup": 1107,
      "after_min_length": 1041,
      "after_spam_filter": 1041,
      "after_timestamp_parse": 1032,
      "final": 672,
      "after_brand_relevance": 672
    },
    "n_posts_final": 672,
    "n_unique_authors_final": 4

In [19]:
# These printouts go straight into your Data Description and Methodology sections.

def comparison_table(hf, rep) -> pd.DataFrame:
    rows = []
    for label, df in [("Hugging Face", hf), ("Replicate", rep)]:
        rows.append({
            "Brand": label,
            "Posts (final)": len(df),
            "Unique authors": df["author_did"].nunique(),
            "Posts/author": round(len(df) / max(df["author_did"].nunique(), 1), 2),
            "Replies (%)": round(df["is_reply"].mean() * 100, 1),
            "Mean likes": round(df["like_count"].mean(), 2),
            "Mean reposts": round(df["repost_count"].mean(), 2),
            "Median post length (clean tokens)": int(df["word_count_clean"].median()),
            "Posts with URLs (%)": round((df["n_urls"] > 0).mean() * 100, 1),
            "Posts with hashtags (%)": round((df["n_hashtags"] > 0).mean() * 100, 1),
        })
    return pd.DataFrame(rows).set_index("Brand")

summary_table = comparison_table(hf_posts, rep_posts)
print(summary_table.T)

# Save for the report
summary_table.T.to_csv(PROJECT_ROOT / "outputs" / "tables" / "01_data_description_summary.csv")

Brand                              Hugging Face  Replicate
Posts (final)                           1608.00     672.00
Unique authors                           770.00     437.00
Posts/author                               2.09       1.54
Replies (%)                               29.20      45.10
Mean likes                                 4.55       2.88
Mean reposts                               0.75       0.38
Median post length (clean tokens)         16.00      18.00
Posts with URLs (%)                       49.80      40.90
Posts with hashtags (%)                   16.70       9.10
